# Operation HealthGuard — Audit Notebook (student starter)

> **Your role.** Chief AI Risk Officer at Klariova Health.
>
> **What this notebook does.** Orchestrates the technical audit (Task 4) by calling your implementations in `lab/`, and then writes results to `governance_portfolio.xlsx` (the single Excel deliverable).
>
> **Where the work happens.**
> - `lab/fairness_metrics.py` — fairness primitives (FNR, FPR, demographic parity, equalized odds)
> - `lab/audit.py` — subgroup, intersectional, and proxy-pathway analysis
> - `lab/shap_analysis.py` — SHAP setup and proxy-feature detection
> - `lab/kri.py` — KRI implementations
>
> **Test your code.** Before running the notebook, run the unit tests:
> ```
> pytest lab/tests/ -v
> ```
> All 34 tests must pass.

## 0 · Imports and load artifacts

In [ ]:
import sys, joblib, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

# Ensure starter/ is on the path so `from lab.<module> import ...` works
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "lab").exists() else HERE.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# Resolve provided artifacts — they live in starter/ regardless of where the
# notebook is being run from. The starter/ directory is where students download
# A1-A6; the solution notebook reads those same artifacts.
# Locate the data/ folder — works whether the notebook is in starter/ or solution/.
def _find_data(start: Path) -> Path:
    for cand in [start / "data", start.parent / "starter" / "data",
                 start.parent.parent / "starter" / "data",
                 start.parent / "data"]:
        if (cand / "healthguard_model.joblib").exists():
            return cand.resolve()
    raise FileNotFoundError("could not locate data/ with healthguard_model.joblib")
DATA = _find_data(ROOT)
MODEL_PATH = DATA / "healthguard_model.joblib"
TRAIN_PATH = DATA / "train.csv"
TEST_PATH  = DATA / "test.csv"

# Portfolio: write to the workbook in the same folder as this notebook.
PORTFOLIO = ROOT / "governance_portfolio.xlsx"
assert PORTFOLIO.exists(), f"Could not find {PORTFOLIO}"

bundle = joblib.load(MODEL_PATH)
model, threshold, schema = bundle["model"], bundle["decision_threshold"], bundle["feature_schema"]
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
X_train, y_train = train[schema["all"]], train["diabetes_outcome"].values
X_test,  y_test  = test[schema["all"]],  test["diabetes_outcome"].values
proba_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= threshold).astype(int)

print(f"Model loaded; threshold = {threshold}")
print(f"Test n={len(test):,} | prevalence={y_test.mean():.3f}")
print(f"Portfolio = {PORTFOLIO}")

## 1 · Sanity-check overall performance

In [ ]:
acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, proba_test)
print(f"  ROC AUC  : {auc:.3f}")
print(f"  Recall   : {rec:.3f}")
print(f"  Precision: {prec:.3f}")
print(f"  F1       : {f1:.3f}")
print(f"  Accuracy : {acc:.3f}")

## 2 · Verify lab/ implementations work

Run the cell below. If you see `NotImplementedError`, your TODO functions in `lab/` are not yet implemented. Switch back to the lab/ files and implement them, then re-run.

In [ ]:
from lab.fairness_metrics import (
    false_negative_rate, false_positive_rate, selection_rate,
    demographic_parity_difference, equalized_odds_difference,
)
from lab.audit import (
    compute_subgroup_metrics, find_most_harmed_subgroup, assess_proxy_pathway,
)
from lab.shap_analysis import setup_explainer, detect_proxy_features, explain_false_negatives
from lab.kri import (
    kri_subgroup_fnr_drift, kri_override_health, kri_inference_anomaly,
)

# Quick validation: compute FNR overall and on Rural + Medicaid cell
overall_fnr = false_negative_rate(y_test, y_pred_test)
print(f"Overall FNR = {overall_fnr:.3f}  (should be ~0.40)")

## 3 · Single-axis fairness — Task 4a

Compute subgroup metrics across each candidate axis. Look at the output and answer (in your written D4 narrative — Tab 04 of the portfolio): which axes show meaningful disparity?

In [ ]:
for col in ["sex", "age_group", "region", "insurance_type"]:
    df = compute_subgroup_metrics(y_test, y_pred_test, sensitive_features=test[col])
    print(f"\n--- {col} ---")
    print(df.round(3).fillna("-"))

## 4 · Intersectional fairness — Task 4b

Slice on `region × insurance_type`. Pay attention to `n_positives` — cells with very small `n_positives` (e.g. <10) are statistically unstable. Use `find_most_harmed_subgroup` with `min_n_positives=20` so the cell you escalate to leadership is one you can defend.

In [ ]:
intersect = compute_subgroup_metrics(
    y_test, y_pred_test,
    sensitive_features=test[["region", "insurance_type"]])
print(intersect.round(3).fillna("-"))

In [ ]:
most_harmed = find_most_harmed_subgroup(intersect, metric_name="FNR", min_n_positives=20)
print("Most-harmed subgroup (n_pos>=20):", most_harmed)

## 5 · Proxy-pathway test — Task 4e

Compare the within-region gap (Rural + Medicaid vs Rural + Private) against the same gap in a reference primary value. If within > reference, it points at insurance as the operative variable.

In [ ]:
verdict = assess_proxy_pathway(
    intersect, primary_axis_value="Rural",
    secondary_axis_low="Medicaid", secondary_axis_high="Private",
    metric_name="FNR")
print(verdict)

## 6 · SHAP analysis — Task 4c, 4d

In [ ]:
np.random.seed(20260425)
bg_idx = np.random.choice(len(X_train), size=200, replace=False)
# setup_explainer returns the BACKGROUND transformed; we transform X_test
# separately for the actual SHAP computation
explainer, X_bg_t, feat_names = setup_explainer(model, X_train.iloc[bg_idx])
X_test_t = model.named_steps["preprocessor"].transform(X_test)
shap_values = explainer.shap_values(X_test_t)
print(f"SHAP values shape: {np.asarray(shap_values).shape}")

In [ ]:
# Programmatic proxy detection — your detect_proxy_features TODO
proxy_candidates = detect_proxy_features(
    shap_values, feat_names,
    protected_feature_substrings=["region", "sex", "age_group"],
    top_k=8,
)
print(proxy_candidates)

In [ ]:
# Local explanations on false negatives in the most-harmed group
most_harmed_mask = (
    (test["region"] == "Rural") &
    (test["insurance_type"].isin(["Medicaid", "Uninsured"]))
).values

local_examples = explain_false_negatives(
    explainer, X_test_t, feat_names,
    y_test, y_pred_test, most_harmed_mask, n_examples=3,
)
for ex in local_examples:
    print(f"\nInstance {ex['instance_index']}: top-3 negative drivers:")
    for f, v in ex["top_negative_features"][:3]:
        print(f"   {f:<35} {v:+.3f}")

## 7 · KRI evaluation — Task 7 driven from your kri.py

In [ ]:
kri1 = kri_subgroup_fnr_drift(y_test, y_pred_test,
                              test[["region", "insurance_type"]],
                              min_n_positives=20)
print("KRI-1 Subgroup FNR Drift:", kri1)

# For KRI-2 we use synthetic baseline numbers (production would feed real telemetry)
kri2 = kri_override_health(override_rate=0.06, structured_share=0.85)
print("\nKRI-2 Override Channel Health:", kri2)

kri3 = kri_inference_anomaly(anomaly_count=0, total_inferences=2000)
print("\nKRI-3 Inference Anomaly Rate:", kri3)

## 8 · Write D4 + KRI status to the Excel portfolio

This cell writes the technical-audit summary into Tab `04_D4_Technical_Audit` and the KRI status into Tab `07_D7_KRI_Dashboard` of `A5_governance_portfolio.xlsx`.

After this, you still need to **fill in the [HAND] tabs by hand** in Excel: D1 (regulatory work), D2 (privacy work), D3 (threat model), D5 (full risk register), D6 (most model card sections), D8 (launch decision). The notebook does NOT do regulatory work for you.

In [ ]:
from openpyxl import load_workbook
wb = load_workbook(PORTFOLIO)

# === Tab 04 — Technical Audit Summary ===
ws = wb["04_D4_Technical_Audit"]
ws["B6"] = round(auc, 3)
ws["B7"] = round(rec, 3)
ws["B8"] = round(prec, 3)
ws["B9"] = round(f1, 3)
ws["B10"] = round(acc, 3)
ws["B11"] = float(threshold)
ws["B12"] = int(len(y_test))
ws["B13"] = round(float(y_test.mean()), 3)

# Region-level table
region_df = compute_subgroup_metrics(y_test, y_pred_test, test["region"])
region_rows = region_df.reset_index().sort_values("subgroup" if "subgroup" in region_df.reset_index().columns else "region")
region_rows.columns = [c if c != region_df.index.name else "region" for c in region_rows.columns]
for i, (_, row) in enumerate(region_rows.iterrows()):
    r = 18 + i
    ws.cell(row=r, column=1, value=str(row.iloc[0]))
    ws.cell(row=r, column=2, value=int(row["n_total"]))
    ws.cell(row=r, column=3, value=int(row["n_positives"]))
    ws.cell(row=r, column=4, value=round(float(row["FNR"]), 3) if pd.notna(row["FNR"]) else None)
    ws.cell(row=r, column=5, value=round(float(row["FPR"]), 3) if pd.notna(row["FPR"]) else None)
    ws.cell(row=r, column=6, value=round(float(row["selection_rate"]), 3))

# Intersectional table — top cells with n_pos >= 5
inter_rows = intersect.reset_index().sort_values("n_positives", ascending=False).head(8)
for i, (_, row) in enumerate(inter_rows.iterrows()):
    r = 27 + i
    ws.cell(row=r, column=1, value=str(row["region"]))
    ws.cell(row=r, column=2, value=str(row["insurance_type"]))
    ws.cell(row=r, column=3, value=int(row["n_total"]))
    ws.cell(row=r, column=4, value=int(row["n_positives"]))
    ws.cell(row=r, column=5, value=round(float(row["FNR"]), 3) if pd.notna(row["FNR"]) else None)
    ws.cell(row=r, column=6, value=round(float(row["selection_rate"]), 3))

# Most-harmed and verdict
ws["B37"] = str(most_harmed["subgroup"]) if most_harmed else "—"
ws["B38"] = round(float(most_harmed["metric_value"]), 3) if most_harmed else None
ws["B39"] = int(most_harmed["n_positives"]) if most_harmed else None
within = verdict.get("within_gap_pp")
ws["B42"] = f"{within:+.1f} pp" if within is not None and not np.isnan(within) else "—"
ws["B43"] = verdict["verdict"]

# Top-K SHAP feature ranking — data rows start at base4+2 = 48
for i, (_, row) in enumerate(proxy_candidates.iterrows()):
    r = 48 + i
    is_proxy = "candidate" if row["mean_shap"] < -0.01 and row["signed_strength"] >= 0.5 else "no"
    ws.cell(row=r, column=1, value=int(row["rank"]))
    ws.cell(row=r, column=2, value=str(row["feature"]))
    ws.cell(row=r, column=3, value=round(float(row["mean_abs_shap"]), 3))
    ws.cell(row=r, column=4, value=round(float(row["mean_shap"]), 3))
    ws.cell(row=r, column=5, value=round(float(row["signed_strength"]), 3))
    ws.cell(row=r, column=6, value=is_proxy)

# === Tab 07 — KRI Dashboard status ===
ws_k = wb["07_D7_KRI_Dashboard"]
for i, kri in enumerate([kri1, kri2, kri3]):
    r = 11 + i
    ws_k.cell(row=r, column=1, value=i + 1)
    ws_k.cell(row=r, column=2, value=kri["kri_id"])
    ws_k.cell(row=r, column=3, value=kri["status"])
    ws_k.cell(row=r, column=4, value=str(kri.get("gap_pp") or kri.get("override_rate") or kri.get("per_1000") or "—"))
    ws_k.cell(row=r, column=5, value=kri.get("note") or kri.get("reason") or "")

wb.save(PORTFOLIO)
print(f"\n✓ Wrote technical-audit + KRI status to {PORTFOLIO.name}")

## 9 · Next steps

You have completed the orchestrated portion of the audit. Now:

1. Open `A5_governance_portfolio.xlsx` and check that Tab `04_D4_Technical_Audit` and Tab `07_D7_KRI_Dashboard` are populated.
2. Fill in the [HAND] tabs:
   - Tab `01_D1_EU_AI_Act` — classification + gap analysis
   - Tab `02_D2_DFD_Compliance` — annotate `A6_data_flow_diagram.png` and populate the obligation table
   - Tab `03_D3_ATLAS` — three TTPs with depth
   - Tab `05_D5_Risk_Register` — synthesize from D1–D4
   - Tab `06_D6_Model_Card` — most sections
   - Tab `08_D8_Launch_Decision` — your recommendation, evidence, conditions, sign-off
3. Run the unit tests one more time to confirm your `lab/` is clean.
4. Submit: this notebook + the lab/ folder + the populated portfolio Excel.

*— end —*